# SPY 0DTE Short Put Credit Spread — Filter-Mode + Rolling Comparison

**Position structure**: sell ATM-ish put, buy further-OTM put as protection. Open at 9:35 ET. Profit target / stop loss measured as % / multiple of credit received. Hard close 3:55 PM ET.

**Regime filters — swept across 4 modes**:
1. **`overnight`** — skip when prior-close → 9:30 open gap < −0.5%
2. **`premarket`** — skip when 9:00 → 9:30 ET move < −0.5%
3. **`either`** — both filters active, skip if EITHER fires (conservative)
4. **`both`** — both filters active, skip only if BOTH fire (less restrictive)

VIX prior-close > 22 filter is ON in all modes.

**Rolling — swept across 2 modes**:
- **`noroll`** — baseline; stop = close at loss
- **`roll2`** — when stop fires intraday, close & re-open at fresh OTM strikes from current spot, up to 2 rolls/day. No roll within 30 min of hard close.

**Sweep grid:** 4 filter × 2 roll × 3 otm × 2 widths × 2 PT × 3 SL = **288 configs**.
- `short_otm_pct`: 0.5%, 1.0%, 1.5%
- `spread_width`: $2, $5
- `profit_target_pct`: 25%, 50% of credit
- `stop_loss_mult`: 1.5×, 2×, 3× credit
- Account: $1,500 start, $50k max-per-trade, $0.85/contract, $0.01 half-spread


## 1. Setup

In [ ]:
import os, sys, subprocess
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
REMOTE = 'https://github.com/coolin815/iron-condor.git'

def sh(cmd, cwd=None):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=120)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f'cmd failed ({r.returncode}): {cmd}')

if not os.path.isdir(REPO):
    sh(f'git clone --depth 50 -b {BRANCH} {REMOTE} {REPO}')
else:
    # Force-sync to remote — drop any local edits/divergence from prior runs.
    sh(f'git -C {REPO} fetch --depth 50 origin {BRANCH}')
    sh(f'git -C {REPO} checkout -f {BRANCH}')
    sh(f'git -C {REPO} reset --hard origin/{BRANCH}')
    sh(f'git -C {REPO} clean -fd')

SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
sh('git log -1 --oneline')
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Validates the spread fill + walk pipeline. Output is a single TradeResult.

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Sweep — past 30 days
~5–10 min. ~21 trading days. First read on WR + per-trade economics.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep

## 4b. Sweep — past 60 days
~10–15 min.

In [ ]:
!python -m iron_condor.cli --days 60 --sweep

## 4c. Sweep — past 365 days
Full year. The real test. Bad-tail days will be in here.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 5. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(40)

## 6. Strike-distance × PT-target grid
Pivot the result by short_otm and PT, holding SL and width fixed at the median values, to see where the strategy is structurally working.

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'(\w+)\|(\w+)\|otm([\d.]+)\|w(\d+)\|pt(\d+)\|sl([\d.]+)x', r['config'])
    if not m: continue
    rows.append({
        'mode': m.group(1),
        'roll': m.group(2),
        'otm_pct': float(m.group(3)),
        'width': int(m.group(4)),
        'pt_pct': int(m.group(5)),
        'sl_mult': float(m.group(6)),
        'return': r['total_return_pct'],
        'win_rate': r['win_rate'],
        'trades': r['n_trades'],
        'avg_pnl': r['avg_net_pnl'],
        'max_dd': r['max_drawdown_pct'],
        'rolled_trades': r.get('rolled_trades', 0),
        'total_rolls': r.get('total_rolls', 0),
    })
grid = pd.DataFrame(rows)

print('=== Filter mode comparison (averaged across roll/PT/SL/OTM/width) ===')
print(grid.groupby('mode').agg(
    avg_return=('return', 'mean'),
    avg_wr=('win_rate', 'mean'),
    avg_trades=('trades', 'mean'),
    avg_max_dd=('max_dd', 'mean'),
).round(2))

print('\n=== Roll mode comparison (averaged across filter/PT/SL/OTM/width) ===')
print(grid.groupby('roll').agg(
    avg_return=('return', 'mean'),
    avg_wr=('win_rate', 'mean'),
    avg_trades=('trades', 'mean'),
    avg_max_dd=('max_dd', 'mean'),
    avg_rolls_per_run=('total_rolls', 'mean'),
).round(2))

print('\n=== Best config per (filter, roll) ===')
best = grid.loc[grid.groupby(['mode', 'roll'])['return'].idxmax()]
print(best[['mode', 'roll', 'otm_pct', 'width', 'pt_pct', 'sl_mult',
            'return', 'win_rate', 'trades', 'max_dd', 'total_rolls']]
      .sort_values('return', ascending=False).round(2).to_string(index=False))

## 7. Top config — exit reason mix
Profit / Stop / Hard-close distribution for the best config. We expect Profit to dominate (that's the credit-spread thesis); Stop should be small but each is bigger than a Profit.

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'hard_close'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Exit reason mix ===')
print(sub['exit_reason'].value_counts())
print('\n=== Avg $ P&L by exit reason ===')
print(taken.groupby('exit_reason')['net_pnl'].agg(['count', 'mean', 'min', 'max']).round(2))
print('\n=== Worst 5 days ===')
print(taken.nsmallest(5, 'net_pnl')[['day', 'exit_reason', 'short_strike', 'long_strike',
                                       'spot_at_entry', 'qty', 'net_pnl']].to_string(index=False))

## 8. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    s = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(s['day']), s['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('0DTE put credit spread — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Per-month breakdown
Where the wins and the losses live across the year.

In [ ]:
trades_all = pd.read_csv('results/sweep_trades.csv')
trades_all['day'] = pd.to_datetime(trades_all['day'])
trades_all['month'] = trades_all['day'].dt.to_period('M').astype(str)
taken_all = trades_all[trades_all['exit_reason'].isin(['profit', 'stop', 'hard_close'])]

top_cfg = summary.iloc[0]['config']
top_taken = taken_all[taken_all['config'] == top_cfg]
print(f'=== Top config ({top_cfg}) per month ===')
if not top_taken.empty:
    print(top_taken.groupby('month').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        worst=('net_pnl', 'min'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2).to_string())